# Task 3: Kafka Topic Design

### 0. Schema thiết kế (message contract)

#### Topic `cpg-nodes` 
| Field | Kiểu | Mô tả |
|:---|:---|:---|
| `id` | string (UUIDv5) | Stable node id — **Kafka key** |
| `type` / `label` | string | Loại AST node (`FunctionDef`, `Name`, ...) |
| `file_path` | string | Đường dẫn tương đối (POSIX) |
| `start_line`, `start_column`, `end_line`, `end_column` | int/null | Vị trí nguồn |
| `code` | string | Snippet |
| `properties` | object | name/value phụ |
| `schema_version` | string | vd. `1.0.0` |
| `event_time` | string | ISO-8601 UTC |

#### Topic `cpg-edges` 
| Field | Kiểu | Mô tả |
|:---|:---|:---|
| `id` | string (UUIDv5) | Stable edge id — **Kafka key** |
| `source_id`, `target_id` | string | Liên kết tới node ids |
| `type` | string | `AST` / `CFG` / `DFG` / `CALL` |
| `properties` | object (optional) | vd. `callee` cho CALL |
| `schema_version`, `event_time` | string | Bắt buộc |

#### Topic `cpg-metadata` 
| Field | Kiểu | Mô tả |
|:---|:---|:---|
| `id` | string (UUIDv5) | Stable file id — **Kafka key** |
| `file_path` | string | Đường dẫn tương đối |
| `size_bytes`, `num_lines` | int | Kích thước file |
| `sha256` | string | Hash nội dung |
| `processed_at`, `status` | string | Thời điểm / SUCCESS |
| `schema_version`, `event_time` | string | Bắt buộc |

#### Topic `cpg-errors` 
| Field | Kiểu | Mô tả |
|:---|:---|:---|
| `id` | string (UUIDv5) | `uuid5(file_path + ":error")` |
| `file_path`, `error_type`, `error_message`, `stack_trace` | string | Chi tiết lỗi |
| `occurred_at`, `schema_version`, `event_time` | string | Bắt buộc |

**Cấu hình broker (lab):** `replication-factor=1`, `partitions=1` (single-broker WSL).


### 0. Cấu hình

Import các module trong `src/task3/` — broker mặc định `localhost:9092` (Docker Compose Task 4 hoặc Kafka WSL).

In [12]:
from src.task3.setup_kafka import setup_kafka
from src.task3.emit import emit
from src.task3.verify import verify
from src.task3.config import KAFKA_BOOTSTRAP, TOPICS, DEMO_LIMIT

print(f"Kafka bootstrap : {KAFKA_BOOTSTRAP}")
print(f"Topics          : {', '.join(TOPICS)}")
print(f"Emit limit      : {DEMO_LIMIT} files")

Kafka bootstrap : 127.0.0.1:9092
Topics          : cpg-nodes, cpg-edges, cpg-metadata, cpg-errors
Emit limit      : 5 files


### 1. Khởi động Kafka & tạo 4 topic

Chạy `src/task3/setup_kafka.py` — tự khởi động ZooKeeper + Kafka (Docker) nếu cần, rồi tạo 4 topic bắt buộc.

In [13]:
def run_task3_setup_kafka():
    ok = setup_kafka()
    if not ok:
        print('[CANH BAO] Chua tao duoc topic — hay khoi dong Kafka (Docker/WSL) roi chay lai cell nay.')

run_task3_setup_kafka()

[INFO] Tao 4 topic bat buoc tren broker Kafka...
  cpg-nodes: da ton tai
  cpg-edges: da ton tai
  cpg-metadata: da ton tai
  cpg-errors: da ton tai

[INFO] kafka-topics --list
  - __consumer_offsets
  - connect-configs
  - connect-offsets
  - connect-status
  - cpg-edges
  - cpg-errors
  - cpg-metadata
  - cpg-nodes
[OK] Du 4 topic Task 3 tren broker.


### 2. Emit sự kiện từ Parser Service lên Kafka (bằng chứng end-to-end)

Gọi `src/task3/emit.py` để phát dữ liệu lên Kafka topics.

In [14]:
def run_task3_emit():
    ok = emit()
    if not ok:
        print('[CANH BAO] Chua emit duoc — dam bao setup_kafka da thanh cong.')

run_task3_emit()

CMD: c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\python.exe E:\BD\src\task2\cpg_parser.py --limit 5 --repo-root E:\BD\peft --discovered-csv E:\BD\output\discovered_files.csv --bootstrap-servers 127.0.0.1:9092 --schema-version 1.0.0
------------------------------------------------------------------------
=== Starting CPG Parser Service ===
Discovered CSV : E:\BD\output\discovered_files.csv
Repo Root      : E:\BD\peft
Kafka Brokers  : 127.0.0.1:9092
Schema Version : 1.0.0
Dry Run Mode   : False

Found 5 source files to parse.
[OK] Connected to Kafka Producer successfully.

[1/5] Processing: docs\source\_config.py ... Parsed and Sent (Nodes: 5, Edges: 6)
[2/5] Processing: examples\adamss_finetuning\glue_adamss_asa_example.py ... Parsed and Sent (Nodes: 2222, Edges: 3935)
[3/5] Processing: examples\adamss_finetuning\glue_adamss_asa_manual_example.py ... Parsed and Sent (Nodes: 2175, Edges: 3841)
[4/5] Processing: examples\adamss_finetuning\image_classification_adamss_asa.py ... 

### 3. Kiểm chứng topic offset & mẫu message trên Kafka

Gọi `src/task3/verify.py` để kiểm tra message count và sample JSON payload từ Kafka.

In [15]:
def run_task3_verify():
    ok = verify()
    if not ok:
        print('[CANH BAO] Chua verify duoc — dam bao emit da thanh cong.')

run_task3_verify()

=== Kafka topic offsets (bang chung ghi thanh cong) ===
  cpg-nodes       begin=0        end=15896    messages=15896
  cpg-edges       begin=0        end=27900    messages=27900
  cpg-metadata    begin=0        end=10       messages=10
  cpg-errors      begin=0        end=0        messages=0
[OK] Saved -> E:\BD\output\task3_offsets.csv

===== SAMPLE from cpg-nodes =====
Kafka key: b5453b47-07b0-5ad7-beb3-42b79396bb76
{
  "id": "b5453b47-07b0-5ad7-beb3-42b79396bb76",
  "type": "Module",
  "label": "Module",
  "file_path": "docs/source/_config.py",
  "ast_path": "Module@0:0-0:0",
  "start_line": null,
  "start_column": null,
  "end_line": null,
  "end_column": null,
  "code": "",
  "properties": {},
  "schema_version": "1.0.0",
  "event_time": "2026-07-24T06:41:16.914129Z"
}

===== SAMPLE from cpg-edges =====
Kafka key: 5a01cb7a-4332-5df7-8964-570b8f7935a6
{
  "id": "5a01cb7a-4332-5df7-8964-570b8f7935a6",
  "source_id": "b5453b47-07b0-5ad7-beb3-42b79396bb76",
  "target_id": "b8030daa-556

### 4. Reflection – Task 3

**What worked**
- 4 topic tách rõ theo loại event → Neo4j chỉ cần subscribe nodes/edges, Spark/Mongo chỉ cần metadata.
- `schema_version` + `event_time` trên mọi message; Kafka key = stable id hỗ trợ sink idempotent ở Task 4.
- `advertised.listeners=PLAINTEXT://localhost:9092` giúp producer Windows nói chuyện được với broker trong WSL.

**What failed**
- Lần đầu broker advertise hostname WSL (`DESKTOP-....localdomain`) khiến client Windows kết nối metadata fail.

**How resolved**
- Sửa `advertised.listeners=PLAINTEXT://127.0.0.1:9092` trong `src/task4/docker-compose.yml` và gom logic tao topic vao `src/task3/setup_kafka.py`.